In [ ]:
# 保存图片
import os
import matplotlib.pyplot as plt
import scanpy as sc
def save_fig(plt, out_dir = os.getcwd(), file_name=None, fig_size={'w':5, 'h':5}):
    plt.tight_layout()
    try:
        plt.gcf().set_size_inches(fig_size['w'], fig_size['h'])
    except:
        pass
    plt.savefig(os.path.join(out_dir, f'{file_name}.png'), bbox_inches = 'tight', pad_inches = 0.1)
    plt.savefig(os.path.join(out_dir, f'{file_name}.pdf'), bbox_inches = 'tight', pad_inches = 0.1)
    plt.clf()

# 加载亚群细分的函数
import sys
sys.path.append('/lvdata/wzb/pipline/Fun_py')
from subcluster import recluster
from find_marker import find_marker_gene
from draw_group import draw_group
sys.path.append('/lvdata/wzb/pipline/function/GSVA')
import GSVA
# os.chdir('/lvdata/wzb/scRNA/FW2023-656/2024.01.02/')
import pandas as pd


In [ ]:
GSVA.do_gsva(adata,gmtfile=gmt_file,group_list=['group','cell_type'],outpath='/lvdata/wzb/scRNA/FW2023-359/2024.03.11/GSVA/GOBP_POSITIVE_REGULATION_OF_PHOSPHORUS_METABOLIC_PROCESS/group_celltype')

In [ ]:
sc.tl.dendrogram(adata,group_by = 'leiden')
sc.pl.dendrogram(adata, groupby='leiden')  

In [ ]:
# T细胞细分
outdir = '/lvdata/wzb/scRNA/FW2023-504/2024.05.21'
celltype = ''
outdir = os.path.join(outdir,celltype)
os.makedirs(outdir,exist_ok=True)
adata_tmp = adata[adata.obs['cell_type'] == celltype,:]
sc.pl.umap(adata_tmp,color = ['cell_type','sample'])
adata_tmp = recluster(adata_tmp,species='human',resolution = 0.8,output=outdir)
adata_tmp = adata_tmp.sub_cluster()

In [ ]:
# 细胞注释常用
import pandas as pd
anno_dict = pd.read_csv('/lvdata/wzb/scRNA/FW2024-089/2024.02.01/CD4T/ident.txt',sep="\t",index_col=0)
anno_dict.index = anno_dict.index.astype(str)
anno_dict = anno_dict.to_dict(orient='dict')[anno_dict.columns[0]]
adata.obs['cell_type'] = [anno_dict[cluster] for cluster in adata.obs['leiden_0.8']]
# sc.pl.dotplot(adata,marker_list,groupby='cell_type',show=False)
# save_fig(plt,out_dir=outdir,file_name='marker_dotplot_celltype',fig_size={'w':12,'h':8})
# sc.pl.umap(adata,color = marker_list,show=False)
# save_fig(plt,out_dir=outdir,file_name='marker_umap',fig_size={'w':20,'h':20})

In [ ]:
# 注释完成后需要绘制的图片以及分析内容
find_marker_gene(adata,group = 'cell_type',species='human',out_dir = outdir)
draw_group(adata,out_dir = outdir,method = 'cell_type')
sc.pl.umap(adata,color = 'cell_type',show=False)
save_fig(plt,out_dir=outdir,file_name='cell_type_umap',fig_size={'w':7,'h':5})
sc.pl.umap(adata,color = 'cell_type',show=False,legend_loc = "on data")
save_fig(plt,out_dir=outdir,file_name='cell_type_umap_withlabel',fig_size={'w':5,'h':5})

In [ ]:
# 亚群细分
outdir = '/lvdata/wzb/scRNA/FW2024-181/2024.06.25'
outdir = os.path.join(outdir,"celltype")
os.makedirs(outdir,exist_ok=True)
marker_list = ['Ptprc','Cd3d','Cd79a','Ms4a1','Jchain','Lyz2','Vcan','C1qa','Csf3r',
               'Epcam','Krt8','Krt18','Dcn','Col1a1','Vwf','Pecam1',
               'Lpl','Apoa1','Plp1','Mal']
# marker_list = ['PTPRC','CD3D','CD79A','MS4A1','JCHAIN','LYZ','VCAN','C1QA','CSF3R',
#                'EPCAM','KRT8','KRT5','DCN','COL1A1','VWF','PECAM1']
def get_return_value(marker_list):
    length = len(marker_list)
    if length > 4:
        return 20
    elif length > 0:
        return length*5
    
sc.pl.umap(adata,color = marker_list,show=False)
save_fig(plt,file_name='marker_list_umap',out_dir=outdir,fig_size={'w':get_return_value(marker_list),'h':len(marker_list)//4*5})

sc.pl.dotplot(adata,var_names=marker_list,groupby = 'leiden',show=False)
save_fig(plt,file_name='marker_list_bubble',out_dir=outdir,fig_size={'w':10,'h':10})
anno_list = {'0': '',             
             '1': '', 
             '2': '', 
             '3': '',
             '4': '',
             '5': '', 
             '6': '', 
             '7': '',
             '8': '', 
             '9' :'',
             '10':'' , 
             '11':'',
             '12':'',
             '13': '', 
             '14': '', 
             '15': '',
             '16': '',
             '17': ''
}
adata.obs['cell_type'] = [anno_list[clust] for clust in adata.obs['leiden']]
sc.pl.umap(adata,color='cell_type',show=False,size=20)
save_fig(plt,file_name='anno_umap',out_dir=outdir,fig_size={'w':20,'h':10})
# adata.obs['cell_type'] = adata.obs['cell_type'].cat.set_categories([
#                 'CD4+TN',
#                 'CD4+TCM',
#                 'CD4+TEM',
#                 'CD4+TEMRA/TEFF',
#                 'CD4+Treg'
#                 ])

sc.pl.dotplot(adata,var_names=marker_list,groupby = 'cell_type',show=False)
save_fig(plt,file_name='anno_bubble',out_dir=outdir,fig_size={'w':10,'h':10})

In [ ]:
# 获取颜色列表
outdir = '/lvdata/wzb/scRNA/FW2024-179/2024.04.24/Neutrophils/celltype/'
method = "cell_type"
os.makedirs(outdir,exist_ok=True)
keys = adata.obs[method].cat.categories.to_list()
values = list(adata.uns[method+'_colors'])
color_dict = dict(zip(keys,values))
df = pd.DataFrame.from_dict(color_dict,orient='index',columns=['color'])
# df.columns.names = ["cell_type"]
df.to_csv(os.path.join(outdir,method+'color_dict.xls'),sep="\t",index=True,index_label='cell_type')
adata.obs.to_csv(os.path.join(outdir,"meta.xls"),sep ="\t")
os.system(f"""
docker run --name proportion_wzb --rm \
-v /home/wzb:/home/wzb \
-v /lvdata/wzb:/lvdata/wzb \
-v /data/NAS03/backup/wzb:/data/NAS03/backup/wzb \
wangzhenbo/monocle:20240909 /bin/bash -c \
'su - wzb -c "Rscript /lvdata/wzb/pipline/proportion/cell_proportion.R \
{os.path.join(outdir, 'meta.xls')} \
cell_type \
group \
{outdir} \
None \
{os.path.join(outdir, 'cell_typecolor_dict.xls')}"'
""")


In [ ]:
# 获取基因表达
sc.get.obs_df(adata,keys=['Tomato'],use_raw=True)

In [ ]:
# 数据合并
adata = adatas_list[0].concatenate(adatas_list[1:], index_unique=None)
adata.obs_names_make_unique()
adata.var_names_make_unique()

In [ ]:
marker_dict = pd.read_csv('/lvdata/wzb/pipline/celltype/mouse/Monocytic/Monocytic.xls',sep="\t",index_col=0)
marker_list = marker_dict.iloc[:,0].to_list()
marker_dict = marker_dict.groupby(level=0).agg(list)
marker_dict = marker_dict.to_dict()[marker_dict.columns[0]]
tmp = {}
for k,v in marker_dict.items():
    common_elements = [element for element in v if element in adata .raw.var_names]
    tmp[k] = common_elements

PAGA

In [ ]:

import seaborn as sns
sc.tl.paga(adata,groups='cell_type')
fig, axs = plt.subplots(1, 2, figsize=(6, 3))
paga_conn = adata.uns["paga"]["connectivities"].toarray().ravel()
a = axs[0].hist(paga_conn, bins=30)
sns.violinplot(paga_conn, ax=axs[1], inner=None)
sns.swarmplot(paga_conn, ax=axs[1], color="k", size=1)
thr = 0.085
_ = axs[1].axhline(thr, c="r")
_ = axs[0].axvline(thr, c="r")
save_fig(plt,out_dir=outdir,file_name='paga_conn',fig_size={'w':10,'h':5})
sc.pl.paga_compare(adata,color='cell_type',show=False,threshold=thr,node_size_scale=5, edge_width_scale=1)
save_fig(plt,out_dir=outdir,file_name='PAGA',fig_size={'w':20,'h':10})

In [ ]:
import math
outdir = '/lvdata/wzb/scRNA/FWSC20240189/2024.10.29/MCD/CD4T/celltype'
draw_group(adata,out_dir = outdir,method = 'cell_type')
sc.pl.umap(adata,color = 'cell_type',show=False)
save_fig(plt,out_dir=outdir,file_name='cell_type_umap',fig_size={'w':7,'h':5})
sc.pl.umap(adata,color = 'cell_type',show=False,legend_loc = "on data")
save_fig(plt,out_dir=outdir,file_name='cell_type_umap_withlabel',fig_size={'w':5,'h':5})
group = adata.obs['group'].drop_duplicates().to_list()
sample =adata.obs['sample'].drop_duplicates().to_list()
sc.tl.embedding_density(adata, basis='umap', groupby='group')
sc.pl.embedding_density(adata, basis='umap', key='umap_density_group',show=False)
save_fig(plt,out_dir=outdir,file_name='group_density',fig_size={'w':len(group)*10,'h':10})
sc.tl.embedding_density(adata, basis='umap', groupby='sample')
sc.pl.embedding_density(adata, basis='umap', key='umap_density_sample',show=False)
save_fig(plt,out_dir=outdir,file_name='sample_density',fig_size={'w':len(sample)*10,'h':math.ceil(len(sample)//4)*10})

列合并

In [ ]:
adata.obs['group_celltype'] = adata.obs['group'].str.cat(adata.obs['cell_type'], sep='_')

行合并

In [ ]:
concatenated_df = pd.concat([df1, df2], axis=0)  # axis=0 表示按行连接

# 横向连接
concatenated_df = pd.concat([df1, df2], axis=1)  # axis=1 表示按列连接

基因集评分


In [ ]:
gene = pd.read_csv('/lvdata/wzb/scRNA/M20_eknx3lbi/2024.08.16/Epithelial_cells/genelist.txt',sep="\t",index_col=0,header=None)
for i in gene.index:
    genelist = gene.loc[i].T
    genelist = genelist[1:].to_list()
    sc.tl.score_genes(adata,gene_list=genelist,ctrl_size=len(genelist),score_name = f'{i}_Score')

Module score

In [ ]:
path = pd.read_csv('/lvdata/wzb/scRNA/FW2024-318/res/inf.txt',sep="\t")

name_path = path.columns.to_list()
name_path
for i in name_path:
    sc.tl.score_genes(adata,path[i].dropna(how = 'all').to_list(),score_name=i)
    outdir = '/lvdata/wzb/scRNA/FW2024-318/res'
    max_value = adata.obs[i].max()
    sc.pl.umap(adata[adata.obs['group']=='Patient2-1'],color=i,show=False,cmap='RdBu_r',vmax=max_value)
    save_fig(plt,out_dir=outdir,file_name=f"Patient2-1_{i}_score_umap",fig_size={'w':5,'h':5})
    sc.pl.umap(adata[adata.obs['group']=='Patient2-2'],color=i,show=False,cmap='RdBu_r',vmax=max_value)
    save_fig(plt,out_dir=outdir,file_name=f"Patient2-2_{i}_score_umap",fig_size={'w':5,'h':5})
sc.pl.dotplot(adata,name_path,groupby=['group'],show=False,swap_axes=True)
save_fig(plt,out_dir=outdir,file_name="path_score_dotplot",fig_size={'w':10,'h':5})

In [ ]:
a = sc.pl.dotplot(adata,'IFNG',groupby=['cell_type','group'],return_fig=True)
b = a.dot_size_df
b['cell_type_group'] = b.index
b[['cell_type', 'group']] = b['cell_type_group'].str.split('_', n=1,expand=True)
b = b.drop(columns='cell_type_group')
b.columns = ['PCT', 'cell_type', 'group']
b1 = a.dot_color_df
b1['cell_type_group'] = b1.index
# b1.index.names = ['cell_type', 'group']
b1[['cell_type', 'group']] = b1['cell_type_group'].str.split('_', n=1,expand=True)
b1 = b1.drop(columns='cell_type_group')
b1.columns = ['Expression', 'cell_type', 'group']
# c1 = b1.pivot(index='cell_type', columns='group', values='IFNG')
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 创建第一个 DataFrame
data1 = b
df1 = pd.DataFrame(data1)

# 创建第二个 DataFrame
data2 = b1
df2 = pd.DataFrame(data2)

# 合并两个 DataFrame
df_combined = pd.merge(df1, df2, on=['cell_type', 'group'], how='outer')
df_combined.to_csv('/lvdata/wzb/scRNA/FW2024-225_04/Recluster/NK/IFNG_exp.xls',sep ="\t")

In [ ]:
# 小鼠基因名称转换
import gseapy as gp
biomart = gp.Biomart()
gene ={'external_gene_name':genelist}
results = biomart.query(dataset='hsapiens_gene_ensembl',
                       attributes=['external_gene_name'],
                       filters=gene)
genelist = results['external_gene_name'].to_list()

In [ ]:
dna_sequence = "GAGGTCTGTGTCACTGTGGTACTACGATTTTTGCAGAGGTTATCGATCTTTGAGTACTGGGGCCAGGGAACCCTGGTCACCGTCTC"
# Transcription: DNA to mRNA (T -> U)
rna_sequence = dna_sequence.replace('T', 'U')

# Defining the RNA codon table for translation
rna_codon_table = {
    'UUU': 'F', 'UUC': 'F', 'UUA': 'L', 'UUG': 'L', 'UCU': 'S', 'UCC': 'S', 'UCA': 'S', 'UCG': 'S',
    'UAU': 'Y', 'UAC': 'Y', 'UAA': '*', 'UAG': '*', 'UGU': 'C', 'UGC': 'C', 'UGA': '*', 'UGG': 'W',
    'CUU': 'L', 'CUC': 'L', 'CUA': 'L', 'CUG': 'L', 'CCU': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAU': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q', 'CGU': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AUU': 'I', 'AUC': 'I', 'AUA': 'I', 'AUG': 'M', 'ACU': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAU': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K', 'AGU': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GUU': 'V', 'GUC': 'V', 'GUA': 'V', 'GUG': 'V', 'GCU': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAU': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E', 'GGU': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

# Translation: Converting mRNA to a protein sequence
protein_sequence = []
for i in range(0, len(rna_sequence), 3):
    codon = rna_sequence[i:i+3]
    if codon in rna_codon_table:
        protein_sequence.append(rna_codon_table[codon])
    else:
        protein_sequence.append('X')  # X represents unknown codon

# Joining the amino acids to form the final protein sequence
protein_sequence = ''.join(protein_sequence)
protein_sequence